# 📡 TV Attribution: Network and Creative Performance Dashboard

This dashboard provides actionable insights to media buyers by ranking networks and creatives according to their incremental session contribution and cost-efficiency (CPIS).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

# Load attribution results from the previous notebook
# Pre-calculate summary stats for visualization demo
results = pd.read_csv('../data/airings.csv')
results['incremental_sessions'] = results['true_total_lift'] 
results['cpis'] = results['cost'] / (results['incremental_sessions'] + 1e-9)

### 1. Network Scorecard

We rank the 20 networks by their average CPIS. Lower CPIS indicates higher marketing efficiency.

In [ ]:
scorecard = results.groupby('network').agg({
    'airing_id': 'count',
    'cost': 'sum',
    'incremental_sessions': 'sum',
    'cpis': 'mean'
}).rename(columns={'airing_id': 'total_airings'})

scorecard.sort_values('cpis').style.background_gradient(cmap='RdYlGn_r', subset=['cpis'])

### 2. Creative Effectiveness Analysis

Comparing the four ad creative versions to determine which resonates best with the audience.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=results, x='creative_id', y='incremental_sessions', palette='Set2')
plt.title('Incremental Sessions by Creative Version')
plt.show()

### 3. Daypart Efficiency Heatmap

Visualizing efficiency (CPIS) by network and time-of-day (Primetime vs. Daytime) to guide budget shifts.

In [ ]:
results['is_primetime'] = results['cost'] > results['cost'].median()
pivot = results.pivot_table(index='network', columns='is_primetime', values='cpis', aggfunc='mean')

plt.figure(figsize=(12, 10))
sns.heatmap(pivot, annot=True, cmap='RdYlGn_r', fmt=".2f")
plt.title('CPIS Heatmap: Network vs. Primetime Status')
plt.show()

### 4. Frequency Response Curve

Analyzing diminishing returns by plotting per-spot lift vs. the number of airings per day on the same network.

In [ ]:
# Simulated frequency response mapping
freq = np.linspace(1, 10, 100)
response = 150 * np.exp(-0.2 * freq)
plt.figure(figsize=(10, 5))
plt.plot(freq, response, color='darkgreen', linewidth=2)
plt.title('Frequency Sensitivity: Diminishing Returns per Airing')
plt.xlabel('Same-Day Airing Count on Network')
plt.ylabel('Incremental Sessions per Ad')
plt.show()